# Demonstration notebook for generation with SurvivEHR

In this notebook we plot marginal transition matrices between event types

In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

!pwd

%load_ext autoreload
%autoreload 2
%env SLURM_NTASKS_PER_NODE=28       # TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/modelling/SurvivEHR/notebooks/CompetingRisk/0_pretraining/2_generation
env: SLURM_NTASKS_PER_NODE=28       # TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import wandb
import pandas as pd
from hydra import compose, initialize
import seaborn as sns
import json
import io
from matplotlib.colors import LogNorm, Normalize
import logging

from SurvivEHR.examples.modelling.SurvivEHR.run_experiment import run
from SurvivEHR.examples.data.map_to_reduced_names import convert_event_names, EVENT_NAME_SHORT_MAP

%env SLURM_NTASKS_PER_NODE=28   

logging.getLogger().setLevel(logging.ERROR)          # Remove deprecation warnings from older dataset format

sns.set(style="ticks", context="notebook")
sns.color_palette("Reds")


env: SLURM_NTASKS_PER_NODE=28


[(0.9950634371395617, 0.8596539792387543, 0.7986620530565167),
 (0.9882352941176471, 0.6866743560169165, 0.5778854286812765),
 (0.9865897731641676, 0.5067281814686659, 0.38123798539023457),
 (0.9570011534025374, 0.3087120338331411, 0.22191464821222606),
 (0.8370472895040368, 0.13394848135332565, 0.13079584775086506),
 (0.6663437139561708, 0.06339100346020761, 0.08641291810841982)]

## Initialise the dataloader used for pre-training

In [3]:
pre_trained_model = "SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1"

# load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../../confs", job_name="causal_metric_testing_notebook"):
    cfg = compose(config_name="config_CompetingRisk11M", 
                  overrides=[# Experiment setup
                             f"experiment.run_id={pre_trained_model}",
                             "experiment.train=False",
                             "experiment.test=False",
                             "experiment.log=False",
                             "data.meta_information_path=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle",
                             "data.min_workers=12",
                            ]
                 )     

model, dm = run(cfg)
print(f"Loaded model with {sum(p.numel() for p in model.parameters())/1e6} M parameters")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Loaded model with 11.20919 M parameters


/rds/bear-apps/2022a/EL8-ice/software/PyTorch-Lightning/2.1.0-foss-2022a-CUDA-11.7.0/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


# Demonstration of how previous diagnosis impacts transition probabilty to the next diagnosis

### Create utility functions

In [4]:
def _get_stratified_next_event_matrix(df, events_of_interest=None):
    """
    Generate a cross-tabulated matrix of next vs. previous events.

    Maps raw event names to shortened labels, builds a contingency
    table of next-event vs. previous-event counts, and optionally
    filters rows and columns to a subset of events of interest.
    """

    df["Next event"] = df["Next event"].map(EVENT_NAME_SHORT_MAP)
    df["Previous event"] = df["Previous event"].map(EVENT_NAME_SHORT_MAP)

    df = pd.crosstab(df['Next event'], df['Previous event'])
    
    # Drop columns and rows 
    if events_of_interest is not None:
        cols_to_keep = df.columns.intersection(events_of_interest[0])
        rows_to_keep = df.index.intersection(events_of_interest[1])
        df = df.loc[rows_to_keep, cols_to_keep]

    return df

def plot_stratified_next_event_matrix(
    df,
    events_of_interest=None, 
    total_threshold=0.0,
    conditional=True, 
    conditional_probability_threshold=0.0,
    max_steps=10, 
    save_name="next_event.png"
):
    """
    Plot a stratified next-event transition matrix as a heatmap.

    Filters generated event sequences by maximum prediction step, 
    builds a next-vs-previous event matrix, applies frequency and 
    probability thresholds, and removes empty rows/columns. The 
    resulting matrix is visualized as a heatmap with counts overlaid.

    Args:
        df (pd.DataFrame): DataFrame containing event generation results.
        events_of_interest (tuple[list, list], optional): Subset of (columns, rows) to include.
        total_threshold (float or int): Minimum count (absolute or proportion of total) to keep.
        conditional (bool): If True, normalize counts to conditional probabilities per column.
        conditional_probability_threshold (float): Minimum probability to display after normalization.
        max_steps (int): Only include events generated before this step.
        save_name (str): File path to save the heatmap image.

    Returns:
        None
    """

    # Filter out by how long into the future we generate
    df = df[df["Generation step"] < max_steps].copy()
    
    df = _get_stratified_next_event_matrix(df, events_of_interest=events_of_interest)

    # Get original statistics
    total_observed = df.values.sum()
    col_totals = df.sum(0).replace(0, np.nan)
    counts_annot = df.copy()                     # preserve counts for annotatiob

    # For combinations occuring fewer than `total_frequency_threshold` times, set to zero
    minimum_threshold = int(total_threshold * total_observed) if type(total_threshold) is float else total_threshold
    df = df.mask(df < minimum_threshold, 0)
            
    # Standardise columns
    if conditional:
        df = df.div(col_totals, axis=1).fillna(0)
        df = df.mask(df < conditional_probability_threshold, 0)
        
    # Remove columns and rows that are all zeros
    nonzero_rows = ~(df == 0).all(1)
    nonzero_cols = (df != 0).any(0)
    df = df.loc[nonzero_rows, nonzero_cols]
    counts_annot = counts_annot.loc[nonzero_rows, nonzero_cols]

    # Set remaining zeros to nan so they don't convolute plot
    df.replace(0, np.nan, inplace=True)
    counts_annot = counts_annot.where(~df.isna())

    n_rows, n_cols = df.shape
    # make the canvas large enough so labels stay readable
    fig_w = min(15, max(8, n_cols * 0.55))   # 0.55 inch per column
    fig_h = max(8, n_rows * 0.45)            # 0.45 inch per row
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), constrained_layout=True)

    # shrink factor: less than 1 and never below 0.35
    shrink = max(0.35, 8 / max(fig_w, fig_h))
    # thickness in points; larger gives a chunkier bar
    cbar_aspect = 30  
    
    label = f"Conditional transition probability $(p > {conditional_probability_threshold};" if conditional else f"Event transitions $("
    label += f"N\geq{minimum_threshold})$"
    
    sns.heatmap(
        df,
        ax=ax,
        cmap="YlOrRd",
        vmin=0,
        annot=counts_annot,
        fmt=".0f",
        cbar_kws={"label": label,
                  "shrink": shrink,
                  "aspect": cbar_aspect,
                  },
        linewidths=0.5,              # thin grid helps visual alignment
    )

    plt.xlabel("Prior event")
    plt.ylabel("Next event")
    plt.grid()
    
    plt.savefig(save_name)
    plt.close()


# Plot subgroup-specific next-event transition matrices

Construct and visualize transition heatmaps between different categories of clinical events (diagnoses, medications, investigations) across multiple datasets.

-----------------------

For each dataset, the code performs the following steps:

1. **Define Event Groups**  
   Extract event names for labs, medications, and diagnoses from the model’s metadata, separating them into distinct subgroups.


2. **Load Generated Predictions**  
    Read the CSV file containing model-generated next-event predictions for the dataset.

4. **Specify Subgroup Comparisons**  
   Define pairs of event categories to compare (e.g., diagnosis → diagnosis, diagnosis → drug, investigation → diagnosis). Map raw names to short labels for readability.

5. **Plot Transition Matrices**   
   For each comparison, call plot_stratified_next_event_matrix to build and plot a conditional next-event transition matrix. Apply thresholds to filter rare or low-probability transitions, then save the heatmap as a PNG image.

The overall function is to highlight how the model predicts transitions between different event subgroups, providing interpretable visual summaries of next-event risks stratified by event type and dataset.

Different thresholds are used for plots that appear in the main manuscript and the Supplmentary Material.

In [5]:
# Get the subgroups of tokens we want to plot by
lab_names = dm.meta_information["measurement_tables"][dm.meta_information["measurement_tables"]["count_obs"] > 0]["event"].to_list()
medication_names = dm.meta_information["measurement_tables"][dm.meta_information["measurement_tables"]["count_obs"] == 0]["event"].to_list()
diagnosis_names = dm.meta_information["diagnosis_table"]["event"].to_list()
max_steps = 3
supplementary_plot = False

datasets = [
    "FineTune_CVD",
    "PreTrain", 
    "FineTune_Hypertension", 
    # "FineTune_MultiMorbidity50+"
]

for dataset in datasets:

    gen_data_path = f'figs/generation/{pre_trained_model}/{dataset}_dataset/'
    
    df = pd.read_csv(gen_data_path + f"next_event_{dataset}.csv")

    for name, events_of_interest in zip(["diagnosis vs diagnosis", "diagnosis vs drug", "investigation vs diagnosis"], 
                                        [[diagnosis_names, diagnosis_names],
                                         [diagnosis_names, medication_names],
                                         [lab_names, diagnosis_names],
                                         ]):
    
        
    
        for i in range(2):
            events_of_interest[i] = [EVENT_NAME_SHORT_MAP[col] if col in EVENT_NAME_SHORT_MAP else col for col in events_of_interest[i]]
    
        plot_stratified_next_event_matrix(
            df,
            events_of_interest=events_of_interest,
            total_threshold=5 if supplementary_plot else 15,
            conditional=True,
            conditional_probability_threshold=0.01 if supplementary_plot else 0.05,
            max_steps=max_steps,
            save_name=gen_data_path+f"generation_matrix_{name}_{max_steps}_SM{supplementary_plot}.png" 
            )
        



# Plot histogram of most frequent next-event transitions

Summarize and visualize the most common preceding events that lead to predicted next events, highlighting their relative frequencies.

-----------------

For each dataset, the function performs the following steps:

1. **Filter Predictions**  
   Restrict the data to events generated within a maximum number of steps (max_steps). Map raw event names to short labels for readability.

2. **Select Top Events**  
   Count the frequency of “previous events” and identify the top-k most common ones. Filter the DataFrame to keep only these high-frequency events.

3. **Prepare Data for Plotting**  
   Convert the filtered events into categorical form to ensure consistent ordering in the plot.

4. **Plot Histogram**  
   Histplot plot of the percentage frequency of the top events along the y-axis.

The overall function is to provide an interpretable summary of which events most often precede model-predicted next events, allowing quick comparison of transition patterns across datasets.

In [6]:
def plot_stratified_next_event_histogram(df, dm, events_of_interest=None, top_k=10, max_steps=10, save_name="next_event.png"):

    # Filter out by how long into the future we generate
    df = df[df["Generation step"] < max_steps].copy()
    
    df["Previous event"] = df["Previous event"].map(EVENT_NAME_SHORT_MAP)
        
    counts = df['Previous event'].value_counts()
    top_labels = counts.head(top_k).index.tolist()

    filtered_df = df[df['Previous event'].isin(top_labels)].copy()
    filtered_df["Previous event"] = pd.Categorical(filtered_df["Previous event"], top_labels)

    fig, axis = plt.subplots(1,1,figsize=(8,5))

    sns.histplot(
        data=filtered_df, 
        y="Previous event", 
        stat="percent",
    )

    plt.ylabel(f"Frequency of events following T2DM diagnosis")
    plt.grid()
    plt.tight_layout()
    
    plt.savefig(save_name)
    plt.close()

In [7]:
datasets = [
    "FineTune_CVD",
    "PreTrain", 
    "FineTune_Hypertension", 
    # "FineTune_MultiMorbidity50+"
]

for dataset in datasets:

    gen_data_path = f'figs/generation/{pre_trained_model}/{dataset}_dataset/'
    
    df = pd.read_csv(gen_data_path + f"next_event_{dataset}.csv")

    for max_steps in [1, 3, 5, 10]:
    
        plot_stratified_next_event_histogram(
            df,
            dm,
            events_of_interest=events_of_interest,
            top_k=20,
            max_steps=max_steps,
            save_name=gen_data_path + f"next_{max_steps}_events_generated_histogram.png",
            )